# Claude API 퀵스타트

uv 기반 환경에서 Anthropic 공식 Python SDK(`anthropic`)로 Claude API를 사용하는 예제입니다.

**사전 준비**: 프로젝트 루트의 `.env` 파일에 `ANTHROPIC_API_KEY`를 넣어주세요.
API 키는 [Claude Console](https://platform.claude.com/)에서 발급받을 수 있습니다.

## 1. 클라이언트 초기화

`.env`에서 API 키를 로드하고 클라이언트를 생성합니다.

In [1]:
import os

import anthropic
from dotenv import load_dotenv

load_dotenv()

assert os.environ.get("ANTHROPIC_API_KEY"), ".env 파일에 ANTHROPIC_API_KEY를 설정해주세요"

client = anthropic.Anthropic()  # ANTHROPIC_API_KEY 환경변수를 자동으로 읽습니다
MODEL = "claude-opus-4-8"
print("클라이언트 준비 완료")

클라이언트 준비 완료


## 2. 기본 메시지 요청

가장 단순한 형태의 단일 요청/응답입니다.

In [2]:
response = client.messages.create(
    model=MODEL,
    max_tokens=16000,
    messages=[
        {"role": "user", "content": "파이썬의 리스트 컴프리헨션을 한 문단으로 설명해줘."}
    ],
)

# content는 블록 리스트이므로 type을 확인한 뒤 접근합니다
for block in response.content:
    if block.type == "text":
        print(block.text)

print("\n--- 토큰 사용량 ---")
print(f"입력: {response.usage.input_tokens}, 출력: {response.usage.output_tokens}")

파이썬의 리스트 컴프리헨션(List Comprehension)은 기존의 리스트나 반복 가능한 객체를 바탕으로 새로운 리스트를 간결하고 직관적으로 생성하는 문법입니다. 기본 형태는 `[표현식 for 요소 in 반복가능객체]`이며, 예를 들어 `[x * 2 for x in range(5)]`는 `[0, 2, 4, 6, 8]`이라는 리스트를 만들어냅니다. 여기에 `if` 조건문을 추가하면 `[x for x in range(10) if x % 2 == 0]`처럼 특정 조건을 만족하는 요소만 걸러낼 수도 있습니다. 이 방식은 `for` 반복문과 `append()`를 사용해 여러 줄로 작성해야 할 코드를 한 줄로 압축할 수 있어 가독성이 높고 실행 속도도 상대적으로 빠르다는 장점이 있지만, 조건이나 표현식이 지나치게 복잡해지면 오히려 코드를 이해하기 어려워질 수 있으므로 적절한 상황에서 사용하는 것이 좋습니다.

--- 토큰 사용량 ---
입력: 38, 출력: 416


## 3. 시스템 프롬프트

`system` 파라미터로 모델의 역할과 응답 방식을 지정할 수 있습니다.

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=16000,
    system="당신은 친절한 파이썬 튜터입니다. 항상 짧은 코드 예제를 포함해서 답변하세요.",
    messages=[{"role": "user", "content": "딕셔너리를 값 기준으로 정렬하려면 어떻게 해?"}],
)

for block in response.content:
    if block.type == "text":
        print(block.text)

## 4. 스트리밍 + 적응형 사고 (adaptive thinking)

긴 출력에는 스트리밍을 사용하는 것이 좋습니다 (타임아웃 방지).
복잡한 작업에는 `thinking={"type": "adaptive"}`로 모델이 스스로 사고 깊이를 조절하게 합니다.
`display: "summarized"`를 지정하면 사고 과정 요약도 함께 받아볼 수 있습니다.

In [ ]:
with client.messages.stream(
    model=MODEL,
    max_tokens=64000,
    thinking={"type": "adaptive", "display": "summarized"},
    output_config={"effort": "high"},
    messages=[
        {"role": "user", "content": "소수를 판별하는 효율적인 파이썬 함수를 작성하고, 시간 복잡도를 분석해줘."}
    ],
) as stream:
    for event in stream:
        if event.type == "content_block_delta":
            if event.delta.type == "thinking_delta":
                print(event.delta.thinking, end="", flush=True)
            elif event.delta.type == "text_delta":
                print(event.delta.text, end="", flush=True)
        elif event.type == "content_block_start":
            if event.content_block.type == "thinking":
                print("\n[사고 과정]\n", flush=True)
            elif event.content_block.type == "text":
                print("\n\n[응답]\n", flush=True)

    final_message = stream.get_final_message()

print(f"\n\n--- 출력 토큰: {final_message.usage.output_tokens} ---")

## 5. 멀티턴 대화

API는 상태를 저장하지 않으므로 매 요청마다 전체 대화 이력을 보냅니다.

In [ ]:
messages = []


def chat(user_input: str) -> str:
    messages.append({"role": "user", "content": user_input})
    response = client.messages.create(
        model=MODEL,
        max_tokens=16000,
        messages=messages,
    )
    answer = next(b.text for b in response.content if b.type == "text")
    messages.append({"role": "assistant", "content": answer})
    return answer


print(chat("내 이름은 지수야. 기억해줘."))
print("---")
print(chat("내 이름이 뭐라고 했지?"))

## 6. 구조화된 출력 (Structured Outputs)

Pydantic 모델을 넘기면 응답이 스키마에 맞는 객체로 검증되어 돌아옵니다.

In [ ]:
from pydantic import BaseModel


class BookInfo(BaseModel):
    title: str
    author: str
    year: int
    genres: list[str]


response = client.messages.parse(
    model=MODEL,
    max_tokens=16000,
    messages=[
        {"role": "user", "content": "조지 오웰의 1984에 대한 정보를 알려줘."}
    ],
    output_format=BookInfo,
)

book = response.parsed_output
print(f"제목: {book.title}")
print(f"저자: {book.author}")
print(f"출간: {book.year}")
print(f"장르: {', '.join(book.genres)}")

## 7. 에러 처리

SDK가 제공하는 타입별 예외 클래스를 구체적인 것부터 순서대로 잡습니다.
429/5xx는 SDK가 기본 2회 자동 재시도합니다.

In [ ]:
try:
    response = client.messages.create(
        model=MODEL,
        max_tokens=1000,
        messages=[{"role": "user", "content": "안녕!"}],
    )
    print(next(b.text for b in response.content if b.type == "text"))
except anthropic.AuthenticationError:
    print("API 키가 유효하지 않습니다. .env 파일을 확인하세요.")
except anthropic.RateLimitError:
    print("요청 한도 초과 — 잠시 후 다시 시도하세요.")
except anthropic.APIStatusError as e:
    print(f"API 오류 ({e.status_code}): {e.message}")
except anthropic.APIConnectionError:
    print("네트워크 연결 오류입니다.")